## Lab 03 – Table Types in Snowflake — ANSWER KEY
**Snowflake Fundamentals 4-day class Lab · © 2026 Innovation In Software Corporation. All rights reserved.**

> **[INSTRUCTOR COPY — Do not distribute to students]**

Topics covered:
1. Permanent tables — full Time Travel and Fail-safe
2. Transient tables — reduced retention, no Fail-safe
3. Temporary tables — session-scoped, invisible to other sessions
4. Dynamic tables — auto-refreshing materialised query results
5. Time Travel — querying past state with BEFORE(STATEMENT =>)
6. Cross-session behaviour of table types
7. Dynamic table refresh and DYNAMIC_TABLE_REFRESH_HISTORY

---
## PART 1 — INSTRUCTOR DEMO
> Students follow along. No typing required until Part 2.

### DEMO 1 | Context Setup

> **[INSTRUCTOR NOTE]**
> A fresh `DEMO_DB` is created for this lab. `IF NOT EXISTS` guards prevent errors on re-runs. The warehouse auto-suspends after 5 minutes to avoid unnecessary spend.

In [ ]:
CREATE DATABASE IF NOT EXISTS DEMO_DB;
CREATE SCHEMA  IF NOT EXISTS DEMO_DB.DEMO_SCHEMA;
USE DATABASE DEMO_DB;
USE SCHEMA   DEMO_SCHEMA;
CREATE WAREHOUSE IF NOT EXISTS DEMO_WH
    WITH WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 300;
USE WAREHOUSE DEMO_WH;

-- Set Retention Time Frame of the DEMO_SCHEMA to 90 days (MAX)
ALTER SCHEMA DEMO_SCHEMA SET DATA_RETENTION_TIME_IN_DAYS = 90;

### DEMO 2 | Create One Table of Each Type

> **[INSTRUCTOR NOTE]**
> All four tables share identical column definitions so their behaviour can be compared side-by-side. The Dynamic Table requires a source table and a TARGET_LAG interval — it cannot be populated with direct INSERTs.
>
> **Key differences:**
>
> | Type | Time Travel | Fail-safe | Session scope |
> |---|---|---|---|
> | Permanent | 0–90 days | 7 days | No |
> | Transient | 0–1 day | None | No |
> | Temporary | 0–1 day | None | **Yes** |
> | Dynamic | 0–90 days | 7 days | No |

In [ ]:
-- 2a. Permanent table
CREATE OR REPLACE TABLE regular_table (
    id         INT,
    name       STRING,
    created_at TIMESTAMP
);

-- 2b. Transient table
CREATE OR REPLACE TRANSIENT TABLE transient_table (
    id         INT,
    name       STRING,
    created_at TIMESTAMP
);

-- 2c. Temporary table (session-scoped)
CREATE OR REPLACE TEMPORARY TABLE temp_table (
    id         INT,
    name       STRING,
    created_at TIMESTAMP
);

-- 2d. Source table for the dynamic table
CREATE OR REPLACE TABLE source_table (
    id         INT,
    name       STRING,
    created_at TIMESTAMP
);

-- 2e. Dynamic table — auto-refreshes from source_table every hour
CREATE OR REPLACE DYNAMIC TABLE dynamic_table
    TARGET_LAG = '1 hour'
    WAREHOUSE  = DEMO_WH
AS
    SELECT id, name, created_at
    FROM source_table
    WHERE created_at >= CURRENT_DATE;

### DEMO 3 | Insert Data and Query All Tables

> **[INSTRUCTOR NOTE]**
> `INSERT ALL` populates multiple tables from a single source query in one statement. A key rule: non-deterministic functions like `CURRENT_TIMESTAMP` must be evaluated in the `SELECT` clause, not inside a `VALUES` constructor in the `FROM` clause. The dynamic table is never written to directly — it is populated automatically by its scheduled refresh from `source_table`.

In [ ]:
INSERT ALL
    INTO regular_table   (id, name, created_at) VALUES (id, name, created_at)
    INTO transient_table (id, name, created_at) VALUES (id, name, created_at)
    INTO temp_table      (id, name, created_at) VALUES (id, name, created_at)
    INTO source_table    (id, name, created_at) VALUES (id, name, created_at)
SELECT id, name, CURRENT_TIMESTAMP AS created_at
FROM VALUES
    (1, 'Alice'),
    (2, 'Bob')
  AS t (id, name);

In [ ]:
SELECT 'regular'   AS table_type, * FROM regular_table
UNION ALL
SELECT 'transient' AS table_type, * FROM transient_table
UNION ALL
SELECT 'temp'      AS table_type, * FROM temp_table
UNION ALL
SELECT 'dynamic'   AS table_type, * FROM dynamic_table;

### DEMO 4 | Time Travel — BEFORE(STATEMENT =>)

> **[INSTRUCTOR NOTE]**
> `SET id = LAST_QUERY_ID()` captures the query ID of the most recent statement. `BEFORE (STATEMENT => $id)` returns the table as it existed before that statement executed. Transient and temporary tables support Time Travel with a maximum of 1 day; permanent tables support up to 90 days.

In [ ]:
SET id = LAST_QUERY_ID();

UPDATE regular_table   SET name = 'Charlie' WHERE id = 1;
UPDATE transient_table SET name = 'Charlie' WHERE id = 1;
UPDATE temp_table      SET name = 'Charlie' WHERE id = 1;

In [ ]:
-- Time Travel: see data before the UPDATE
SELECT 'regular'   AS tbl, * FROM regular_table   BEFORE (STATEMENT => $id)
UNION ALL
SELECT 'transient' AS tbl, * FROM transient_table BEFORE (STATEMENT => $id)
UNION ALL
SELECT 'temp'      AS tbl, * FROM temp_table      BEFORE (STATEMENT => $id);

### DEMO 5 | Dynamic Table Refresh

> **[INSTRUCTOR NOTE]**
> A manual `ALTER DYNAMIC TABLE ... REFRESH` triggers an immediate refresh outside the scheduled TARGET_LAG window. `DYNAMIC_TABLE_REFRESH_HISTORY()` shows every refresh run with its start time, status, and rows processed.

In [ ]:
INSERT INTO source_table VALUES (3, 'Charlie', CURRENT_TIMESTAMP);

ALTER DYNAMIC TABLE dynamic_table REFRESH;

In [ ]:
%%sql -r dataframe_8
SELECT * FROM dynamic_table;

In [ ]:
SELECT *
FROM TABLE(INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY())
WHERE NAME = 'DYNAMIC_TABLE';

### DEMO 6 | Cross-Session Behaviour

> **[INSTRUCTOR NOTE]**
> Open a **new worksheet or notebook** and run the four SELECT statements below. The temporary table does not exist in the new session — the query raises "Table does not exist". Permanent, transient, and dynamic tables persist across sessions.

In [ ]:
-- Run this block in a NEW separate worksheet/session:
USE ROLE accountadmin;
USE SCHEMA DEMO_DB.DEMO_SCHEMA;

SELECT * FROM regular_table;    -- succeeds
SELECT * FROM transient_table;  -- succeeds
SELECT * FROM temp_table;       -- fails: table does not exist
SELECT * FROM dynamic_table;    -- succeeds

### DEMO CLEANUP

> Drop objects created during the demo.

In [ ]:
-- DROP DATABASE IF EXISTS DEMO_DB;   -- uncomment if you are not continuing to the next lab
-- DROP WAREHOUSE IF EXISTS DEMO_WH;  -- uncomment if you are not continuing to the next lab
USE WAREHOUSE COMPUTE_WH;

---
## PART 2 — STUDENT EXERCISES — ANSWER KEY
> All exercises create objects in `DEMO_DB.DEMO_SCHEMA`. Clean-up steps are provided at the end.

### EXERCISE 1 | Create a Transient Table and Verify Retention

**Task:** Create a transient table called `products` with columns `id INT`, `sku STRING`, `price FLOAT`. Insert 3 rows. Then run `SHOW TABLES LIKE 'PRODUCTS'` and use the `->>` operator to return only `"name"` and `"retention_time"`. What retention value do you see compared to `regular_table`?

> **[ANSWER NOTE]** Transient tables have a maximum `DATA_RETENTION_TIME_IN_DAYS` of 1. Permanent tables inherit the schema/account setting — in this lab the schema was set to 90 days, so `regular_table` shows `retention_time = 90`. There is no Fail-safe period for transient tables, which reduces storage costs.

In [ ]:
CREATE OR REPLACE TRANSIENT TABLE products (
    id    INT,
    sku   STRING,
    price FLOAT
);

INSERT INTO products VALUES
    (1, 'SKU-001', 9.99),
    (2, 'SKU-002', 24.99),
    (3, 'SKU-003', 4.49);

SHOW TABLES LIKE 'PRODUCTS'
->> SELECT "name", "retention_time" FROM $1;

### EXERCISE 2 | Time Travel on a Permanent Table

**Task:** In `regular_table`, update the `name` of `id = 2` to `'Diana'`. Capture the query ID, then use `BEFORE (STATEMENT => $id)` to retrieve the row for `id = 2` as it existed before the update. Return `id`, `name` (historical), and `name` (current) side by side.

> **[ANSWER NOTE]** The key pattern is: capture `LAST_QUERY_ID()` into a session variable immediately after the DML, then reference `$id` in the Time Travel clause. The JOIN on `id` produces a before/after comparison in a single result set.

In [ ]:
UPDATE regular_table SET name = 'Diana' WHERE id = 2;

SET id = LAST_QUERY_ID();

SELECT
    cur.id,
    hist.name AS name_before,
    cur.name  AS name_after
FROM regular_table                           AS cur
JOIN regular_table BEFORE (STATEMENT => $id) AS hist
  ON cur.id = hist.id
WHERE cur.id = 2;

### EXERCISE 3 | Dynamic Table with a Different Source

**Task:** Create a table called `events` with columns `event_id INT`, `category STRING`, `event_date DATE`. Insert 4 rows across two categories. Create a dynamic table called `events_summary` with `TARGET_LAG = '1 minute'` that aggregates `COUNT(*) AS event_count` grouped by `category`. Manually refresh it and query the result.

> **[ANSWER NOTE]** Dynamic tables are declarative — you define the query and Snowflake keeps the result current. `TARGET_LAG = '1 minute'` sets the maximum staleness. `ALTER DYNAMIC TABLE ... REFRESH` forces an immediate refresh for demo purposes.

In [ ]:
CREATE OR REPLACE TABLE events (
    event_id   INT,
    category   STRING,
    event_date DATE
);

INSERT INTO events VALUES
    (1, 'Webinar',  '2026-03-01'),
    (2, 'Workshop', '2026-03-02'),
    (3, 'Webinar',  '2026-03-03'),
    (4, 'Workshop', '2026-03-04');

CREATE OR REPLACE DYNAMIC TABLE events_summary
    TARGET_LAG = '1 minute'
    WAREHOUSE  = DEMO_WH
AS
    SELECT category, COUNT(*) AS event_count
    FROM events
    GROUP BY category;

ALTER DYNAMIC TABLE events_summary REFRESH;

SELECT * FROM events_summary;

### EXERCISE 4 | CHALLENGE — Temporary Table Visibility

**Task:**
- A) Create a temporary table called `session_log` with columns `action STRING`, `logged_at TIMESTAMP`. Insert 2 rows.
- B) Confirm `SELECT * FROM session_log` succeeds in this notebook.
- C) Open a **new worksheet** in a separate browser tab, switch to `DEMO_DB.DEMO_SCHEMA`, and try `SELECT * FROM session_log`. Capture the error message in a comment below.

Explain in a comment why the temporary table does not appear in the new session.

> **[ANSWER NOTE]** Temporary tables are **session-scoped** — they exist only within the session that created them and are automatically dropped when that session ends. The new worksheet creates a new session with no visibility into temporary objects from the original session. The error will be: `"Object 'SESSION_LOG' does not exist or not authorized."`

In [ ]:
-- Tasks A & B – create and verify the temporary table
CREATE OR REPLACE TEMPORARY TABLE session_log (
    action    STRING,
    logged_at TIMESTAMP
);

INSERT INTO session_log VALUES
    ('login', CURRENT_TIMESTAMP),
    ('query', CURRENT_TIMESTAMP);

SELECT * FROM session_log;

In [ ]:
-- Task C answer:
-- When running SELECT * FROM session_log in a NEW worksheet, the error is:
-- "Object 'SESSION_LOG' does not exist or not authorized."
--
-- Explanation: Temporary tables are session-scoped. They exist only in the session
-- that created them and are invisible to all other sessions. Each Snowflake worksheet
-- or notebook runs in its own independent session, so the new worksheet cannot see
-- temporary objects from this session.

### EXERCISE CLEANUP

In [ ]:
DROP TABLE         IF EXISTS DEMO_DB.DEMO_SCHEMA.products;
DROP TABLE         IF EXISTS DEMO_DB.DEMO_SCHEMA.events;
DROP DYNAMIC TABLE IF EXISTS DEMO_DB.DEMO_SCHEMA.events_summary;
DROP TABLE         IF EXISTS DEMO_DB.DEMO_SCHEMA.session_log;